In [2]:
import pandas as pd
import numpy as np
import json
import os

def run_pipeline(csv_path="syria_rain_cereals_merged.csv", output_json="data.json"):
    print(f"Loading dataset from: {csv_path}")

    if not os.path.exists(csv_path):
        if os.path.exists("data/processed/syria_rain_cereals_merged.csv"):
            csv_path = "data/processed/syria_rain_cereals_merged.csv"
        else:
            raise FileNotFoundError(f"File not found at {csv_path}. Please upload it first.")

    df = pd.read_csv(csv_path)
    df = df[(df["year"] >= 1963) & (df["year"] <= 2024)].sort_values("year").reset_index(drop=True)

    # 1. Top
    top10 = df.nlargest(10, "rain_mm").sort_values("rain_mm", ascending=False).reset_index(drop=True)
    bot10 = df.nsmallest(10, "rain_mm").sort_values("rain_mm", ascending=True).reset_index(drop=True)

    top10_list = top10[["year", "rain_mm", "production_t", "yield_kg_ha"]].to_dict(orient="records")
    bot10_list = bot10[["year", "rain_mm", "production_t", "yield_kg_ha"]].to_dict(orient="records")

    # 2. Drought Analysis
    drought_years = set(df[df['drought'] == True]['year'].values)
    sorted_drought_years = sorted(list(drought_years))

    single_droughts = []
    consecutive_spells = []

    current_spell = []
    for yr in sorted_drought_years:
        if not current_spell:
            current_spell.append(yr)
        else:
            if yr == current_spell[-1] + 1:
                current_spell.append(yr)
            else:
                if len(current_spell) == 1:
                    single_droughts.append(current_spell[0])
                else:
                    consecutive_spells.append(current_spell)
                current_spell = [yr]
    if current_spell:
        if len(current_spell) == 1:
            single_droughts.append(current_spell[0])
        else:
            consecutive_spells.append(current_spell)

    # Drought Recovery
    single_recovery_list = []
    for yr in single_droughts:
        next_yr = yr + 1
        if next_yr in df['year'].values:
            d_row = df[df['year'] == yr].iloc[0]
            r_row = df[df['year'] == next_yr].iloc[0]

            yield_rec = ((r_row['yield_kg_ha'] - d_row['yield_kg_ha']) / d_row['yield_kg_ha']) * 100
            prod_rec = ((r_row['production_t'] - d_row['production_t']) / d_row['production_t']) * 100

            single_recovery_list.append({
                "drought_year": int(yr),
                "drought_rain_mm": float(d_row['rain_mm']),
                "recovery_year": int(next_yr),
                "recovery_rain_mm": float(r_row['rain_mm']),
                "yield_recovery_pct": round(float(yield_rec), 2),
                "prod_recovery_pct": round(float(prod_rec), 2)
            })

    # Consecutive Drought Recovery
    consecutive_summary = []
    for spell in consecutive_spells:
        start_yr = spell[0]
        end_yr = spell[-1]
        next_yr = end_yr + 1

        last_drought_row = df[df['year'] == end_yr].iloc[0]

        if next_yr in df['year'].values:
            r_row = df[df['year'] == next_yr].iloc[0]
            yield_rec = ((r_row['yield_kg_ha'] - last_drought_row['yield_kg_ha']) / last_drought_row['yield_kg_ha']) * 100
            prod_rec = ((r_row['production_t'] - last_drought_row['production_t']) / last_drought_row['production_t']) * 100

            consecutive_summary.append({
                "spell_years": [int(y) for y in spell],
                "spell_length": len(spell),
                "last_drought_year": int(end_yr),
                "recovery_year": int(next_yr),
                "recovery_rain_mm": float(r_row['rain_mm']),
                "yield_recovery_pct": round(float(yield_rec), 2),
                "prod_recovery_pct": round(float(prod_rec), 2)
            })
    avg_single_yield_rec = np.mean([x['yield_recovery_pct'] for x in single_recovery_list]) if single_recovery_list else 0
    avg_consec_yield_rec = np.mean([x['yield_recovery_pct'] for x in consecutive_summary]) if consecutive_summary else 0

    drought_comparison = {
        "avg_single_drought_yield_recovery_pct": round(float(avg_single_yield_rec), 2),
        "avg_consecutive_drought_yield_recovery_pct": round(float(avg_consec_yield_rec), 2),
        "total_single_drought_events": len(single_droughts),
        "total_consecutive_drought_spells": len(consecutive_spells),
        "consecutive_spells_details": consecutive_summary,
        "single_drought_details": single_recovery_list
    }

    # 3. Baseline Statistics
    baseline = df[(df["year"] >= 1981) & (df["year"] <= 2010)]
    stats = {
        "baseline_period": "1981-2010",
        "baseline_rain_mean_mm": round(float(baseline["rain_mm"].mean()), 2),
        "baseline_prod_mean_tonnes": round(float(baseline["production_t"].mean()), 2),
        "top10_rain_avg_mm": round(float(top10["rain_mm"].mean()), 2),
        "top10_prod_avg_tonnes": round(float(top10["production_t"].mean()), 2),
        "bot10_rain_avg_mm": round(float(bot10["rain_mm"].mean()), 2),
        "bot10_prod_avg_tonnes": round(float(bot10["production_t"].mean()), 2),
    }

    # 4. Final Output
    final_payload = {
        "summary_stats": stats,
        "timeseries_data": df.to_dict(orient="records"),
        "top_10_wettest": top10_list,
        "bottom_10_driest": bot10_list,
        "drought_recovery_comparison": drought_comparison
    }

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(final_payload, f, ensure_ascii=False, indent=2)

    print(f"saved to '{output_json}'")

run_pipeline()

Loading dataset from: syria_rain_cereals_merged.csv
saved to 'data.json'
